In [0]:
import os
import json
import urllib.request
import urllib.parse
from datetime import date, timedelta
from delta.tables import DeltaTable
from pyspark.sql import functions as F


In [0]:
# from audit import (
#     iniciar_auditoria,
#     obter_metricas_delta,
#     finalizar_auditoria,
# )

In [0]:
# # Valores padrão permitem testar o notebook manualmente
# dbutils.widgets.text("run_id", "MANUAL")
# dbutils.widgets.text("job_name", "job_cambio_manual")
# dbutils.widgets.text("task_name", "bronze_manual")

# run_id = dbutils.widgets.get("run_id")
# job_name = dbutils.widgets.get("job_name")
# task_name = dbutils.widgets.get("task_name")

# print(f"run_id: {run_id}")
# print(f"job_name: {job_name}")
# print(f"task_name: {task_name}")

In [0]:
from pyspark.sql import functions as F

catalogo = "databricks_cata_managed"
volume_landing = "cambio_ptax_raw_files"

# Janela móvel curta para carga incremental
data_fim = date.today()
data_inicio = data_fim 

data_inicio_api = data_inicio.strftime("%m-%d-%Y")
data_fim_api = data_fim.strftime("%m-%d-%Y")

data_inicio_ref = data_inicio.strftime("%Y-%m-%d")
data_fim_ref = data_fim.strftime("%Y-%m-%d")

batch_id = f"{data_inicio_ref}_{data_fim_ref}".replace("-", "")

tabela_bronze = f"{catalogo}.bronze.cambio_ptax_raw"
landing_dir = f"/Volumes/{catalogo}/landing/{volume_landing}/batch_{batch_id}"

tabela_bronze = f"{catalogo}.bronze.cambio_ptax_raw"

print(f"Lendo landing: {landing_dir}")
print(f"Salvando bronze: {tabela_bronze}")


In [0]:
df_bronze = (
    spark.read
    .option("multiLine", "true")
    .json(f"{landing_dir}/*.json")
    .withColumn("_arquivo_lido", F.col("_metadata.file_path"))
    .withColumn("_data_ingestao", F.current_timestamp())
    .withColumn("_batch_processamento", F.lit(batch_id))
    .withColumn("_camada", F.lit("bronze"))
    .withColumn("_origem", F.lit("API PTAX - Banco Central"))
)

display(df_bronze)

In [0]:
# try:
#     # Seu código de leitura
#     df_bronze = (
#         spark.read
#         .option("multiline", "true")
#         .json(caminho_landing)
#     )

#     linhas_lidas = df_bronze.count()

#     # Seu WRITE ou MERGE
#     if not spark.catalog.tableExists(tabela_destino):
#         (
#             df_bronze.write
#             .format("delta")
#             .mode("overwrite")
#             .option("mergeSchema", "true")
#             .saveAsTable(tabela_destino)
#         )
#     else:
#         delta_bronze = DeltaTable.forName(
#             spark,
#             tabela_destino,
#         )

#         (
#             delta_bronze.alias("t")
#             .merge(
#                 df_bronze.alias("s"),
#                 """
#                 t.batch_id = s.batch_id
#                 AND t.moeda = s.moeda
#                 """,
#             )
#             .whenMatchedUpdateAll()
#             .whenNotMatchedInsertAll()
#             .execute()
#         )

#     metricas = obter_metricas_delta(
#         spark=spark,
#         tabela_destino=tabela_destino,
#     )

#     finalizar_auditoria(
#         spark=spark,
#         run_id=run_id,
#         task_name=task_name,
#         tabela_destino=tabela_destino,
#         inicio_execucao=inicio_execucao,
#         status="SUCCEEDED",
#         linhas_lidas=linhas_lidas,
#         linhas_inseridas=metricas["linhas_inseridas"],
#         linhas_atualizadas=metricas["linhas_atualizadas"],
#         linhas_rejeitadas=0,
#     )

# except Exception as erro:
#     finalizar_auditoria(
#         spark=spark,
#         run_id=run_id,
#         task_name=task_name,
#         tabela_destino=tabela_destino,
#         inicio_execucao=inicio_execucao,
#         status="FAILED",
#         linhas_lidas=locals().get("linhas_lidas", 0),
#         linhas_inseridas=0,
#         linhas_atualizadas=0,
#         linhas_rejeitadas=0,
#         mensagem_erro=str(erro),
#     )

#     raise

In [0]:
# from datetime import datetime
# from zoneinfo import ZoneInfo

# from delta.tables import DeltaTable
# from pyspark.sql import functions as F
# from pyspark.sql.types import (
#     ArrayType,
#     MapType,
#     StructType,
# )

# from audit import (
#     iniciar_auditoria,
#     obter_metricas_delta,
#     finalizar_auditoria,
# )


# # ============================================================
# # Parâmetros do Databricks Job
# # ============================================================

# dbutils.widgets.text("run_id", "MANUAL")
# dbutils.widgets.text("job_name", "job_cambio_manual")
# dbutils.widgets.text("task_name", "bronze_manual")

# run_id_parametro = dbutils.widgets.get("run_id")
# job_name = dbutils.widgets.get("job_name")
# task_name = dbutils.widgets.get("task_name")


# # Gera um ID diferente para cada execução manual.
# if run_id_parametro == "MANUAL":
#     run_id = (
#         "MANUAL_"
#         + datetime.now(
#             ZoneInfo("America/Maceio")
#         ).strftime("%Y%m%d_%H%M%S")
#     )
# else:
#     run_id = run_id_parametro


# print(f"run_id: {run_id}")
# print(f"job_name: {job_name}")
# print(f"task_name: {task_name}")


# # ============================================================
# # Configurações
# # ============================================================

# catalogo = "databricks_cata_managed"
# volume_landing = "cambio_ptax_raw_files"

# camada = "bronze"

# tabela_destino = (
#     f"{catalogo}.bronze.cambio_ptax_raw"
# )

# volume_origem = (
#     f"/Volumes/{catalogo}/landing/{volume_landing}"
# )


# # ============================================================
# # Definição do batch
# # ============================================================

# # Usa o horário de Maceió em vez do horário UTC do servidor.
# data_processamento = datetime.now(
#     ZoneInfo("America/Maceio")
# ).date()

# data_inicio = data_processamento
# data_fim = data_processamento

# data_inicio_ref = data_inicio.strftime("%Y-%m-%d")
# data_fim_ref = data_fim.strftime("%Y-%m-%d")

# batch_id = (
#     f"{data_inicio_ref}_{data_fim_ref}"
#     .replace("-", "")
# )

# landing_dir = (
#     f"{volume_origem}/batch_{batch_id}"
# )


# print(f"Lendo landing: {landing_dir}")
# print(f"Salvando bronze: {tabela_destino}")
# print(f"Batch ID: {batch_id}")


# # ============================================================
# # Função para alinhar o schema ao destino
# # ============================================================

# def alinhar_schema_destino(
#     df_origem,
#     tabela_delta: str,
# ):
#     """
#     Converte as colunas do DataFrame para os mesmos tipos
#     existentes na tabela Delta de destino.

#     Para ARRAY, STRUCT e MAP, utiliza conversão por JSON,
#     evitando erro de cast em estruturas complexas.
#     """

#     schema_destino = spark.table(
#         tabela_delta
#     ).schema

#     df_alinhado = df_origem

#     for campo in schema_destino.fields:
#         nome_coluna = campo.name
#         tipo_destino = campo.dataType

#         if nome_coluna not in df_alinhado.columns:
#             df_alinhado = df_alinhado.withColumn(
#                 nome_coluna,
#                 F.lit(None).cast(tipo_destino),
#             )

#         elif isinstance(
#             tipo_destino,
#             (ArrayType, StructType, MapType),
#         ):
#             df_alinhado = df_alinhado.withColumn(
#                 nome_coluna,
#                 F.from_json(
#                     F.to_json(F.col(nome_coluna)),
#                     tipo_destino,
#                 ),
#             )

#         else:
#             df_alinhado = df_alinhado.withColumn(
#                 nome_coluna,
#                 F.col(nome_coluna).cast(
#                     tipo_destino
#                 ),
#             )

#     # Mantém exatamente as colunas e a ordem da tabela.
#     return df_alinhado.select(
#         *[
#             campo.name
#             for campo in schema_destino.fields
#         ]
#     )


# # ============================================================
# # Variáveis da execução
# # ============================================================

# inicio_execucao = None

# linhas_lidas = 0
# linhas_inseridas = 0
# linhas_atualizadas = 0
# linhas_rejeitadas = 0


# try:
#     # ========================================================
#     # Início da auditoria
#     # ========================================================

#     inicio_execucao = iniciar_auditoria(
#         spark=spark,
#         run_id=run_id,
#         job_name=job_name,
#         task_name=task_name,
#         camada=camada,
#         tabela_origem=landing_dir,
#         tabela_destino=tabela_destino,
#         batch_id=batch_id,
#     )


#     # ========================================================
#     # Leitura da Landing
#     # ========================================================

#     df_bronze = (
#         spark.read
#         .option("multiLine", "true")
#         .json(f"{landing_dir}/*.json")
#         .withColumn(
#             "_arquivo_lido",
#             F.col("_metadata.file_path"),
#         )
#         .withColumn(
#             "_data_ingestao",
#             F.current_timestamp(),
#         )
#         .withColumn(
#             "_batch_processamento",
#             F.lit(batch_id),
#         )
#         .withColumn(
#             "_camada",
#             F.lit(camada),
#         )
#         .withColumn(
#             "_origem",
#             F.lit(
#                 "API PTAX - Banco Central"
#             ),
#         )
#     )


#     linhas_lidas = df_bronze.count()

#     print(f"Linhas lidas: {linhas_lidas}")


#     if linhas_lidas == 0:
#         raise ValueError(
#             f"Nenhum registro encontrado em "
#             f"{landing_dir}"
#         )


#     # ========================================================
#     # Validação das chaves do MERGE
#     # ========================================================

#     chaves_merge = [
#         "_batch_processamento",
#         "moeda",
#     ]

#     colunas_faltantes = [
#         coluna
#         for coluna in chaves_merge
#         if coluna not in df_bronze.columns
#     ]

#     if colunas_faltantes:
#         raise ValueError(
#             "Colunas necessárias para o MERGE "
#             f"não encontradas: {colunas_faltantes}. "
#             f"Colunas disponíveis: {df_bronze.columns}"
#         )


#     # ========================================================
#     # Escrita na Bronze
#     # ========================================================

#     if not spark.catalog.tableExists(
#         tabela_destino
#     ):
#         (
#             df_bronze.write
#             .format("delta")
#             .mode("overwrite")
#             .option("mergeSchema", "true")
#             .saveAsTable(tabela_destino)
#         )

#     else:
#         # Resolve diferenças de tipos, especialmente
#         # na coluna ARRAY<STRUCT> chamada registros.
#         df_bronze_alinhado = alinhar_schema_destino(
#             df_origem=df_bronze,
#             tabela_delta=tabela_destino,
#         )

#         delta_bronze = DeltaTable.forName(
#             spark,
#             tabela_destino,
#         )

#         (
#             delta_bronze.alias("t")
#             .merge(
#                 df_bronze_alinhado.alias("s"),
#                 """
#                 t._batch_processamento =
#                     s._batch_processamento
#                 AND t.moeda = s.moeda
#                 """,
#             )
#             .whenMatchedUpdateAll()
#             .whenNotMatchedInsertAll()
#             .execute()
#         )


#     # ========================================================
#     # Métricas Delta
#     # ========================================================

#     metricas = obter_metricas_delta(
#         spark=spark,
#         tabela_destino=tabela_destino,
#     )

#     linhas_inseridas = metricas[
#         "linhas_inseridas"
#     ]

#     linhas_atualizadas = metricas[
#         "linhas_atualizadas"
#     ]


#     print(f"Linhas lidas: {linhas_lidas}")
#     print(f"Linhas inseridas: {linhas_inseridas}")
#     print(f"Linhas atualizadas: {linhas_atualizadas}")
#     print(f"Linhas rejeitadas: {linhas_rejeitadas}")


#     # ========================================================
#     # Auditoria de sucesso
#     # ========================================================

#     finalizar_auditoria(
#         spark=spark,
#         run_id=run_id,
#         task_name=task_name,
#         tabela_destino=tabela_destino,
#         inicio_execucao=inicio_execucao,
#         status="SUCCEEDED",
#         linhas_lidas=linhas_lidas,
#         linhas_inseridas=linhas_inseridas,
#         linhas_atualizadas=linhas_atualizadas,
#         linhas_rejeitadas=linhas_rejeitadas,
#     )


# except Exception as erro:
#     # ========================================================
#     # Auditoria de falha
#     # ========================================================

#     if inicio_execucao is not None:
#         finalizar_auditoria(
#             spark=spark,
#             run_id=run_id,
#             task_name=task_name,
#             tabela_destino=tabela_destino,
#             inicio_execucao=inicio_execucao,
#             status="FAILED",
#             linhas_lidas=linhas_lidas,
#             linhas_inseridas=linhas_inseridas,
#             linhas_atualizadas=linhas_atualizadas,
#             linhas_rejeitadas=linhas_rejeitadas,
#             mensagem_erro=str(erro),
#         )

#     raise